# Pre-requisites

Start containers:

    docker compose up -d notebook kafka zookeeper

Create a topic in Kafka container:

    # enter kafka container to manage kafka server and topics
    docker exec -it kafka bash
    
    # in kafka container
    
    # create new topic
    kafka-topics --create --topic my-topic --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
    
    # list topics
    kafka-topics --list --bootstrap-server localhost:9092
    
    # describe topic
    kafka-topics --describe --topic my-topic --bootstrap-server localhost:9092

Make sure required packages are installed:


In [2]:
%pip install kafka-python requests

Note: you may need to restart the kernel to use updated packages.


# Kafka Demo

KafkaProducer requires key and value as bytes, not strings

See mongodb_documentation.docx document to see the explanations for the docker-compose.yml

In [1]:
import time
from pprint import pprint

In [2]:
# some global variables:

MY_TOPIC = "my-topics"
BOOTSTRAP_SERVER = 'host.docker.internal:9093'

### Docs: [KafkaProducer](https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html)

In [4]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVER,
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    value_serializer=lambda v: v.encode('utf-8')
)

In [24]:
# Send value only
producer.send(MY_TOPIC, value='Hello Kafka!')
producer.flush()  # flush to push messages out of the buffer
print("Message sent!")

# Send key-value pair
producer.send(MY_TOPIC, key='order123', value='Order Placed')
producer.flush()
print("Key-value message sent!")

# Error handling example
try:
    producer.send(MY_TOPIC, value='Error Test')
    producer.flush()
except Exception as e:
    print(f"Error occurred: {e}")


# Send message with key = "STOP" to be caught in consomer loop
producer.send(MY_TOPIC, key='STOP', value='STOP')
producer.flush()
print("STOP message sent!")


Message sent!
Key-value message sent!
STOP message sent!


### Docs: [KafkaConsumer](https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html)

In [32]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    MY_TOPIC,
    bootstrap_servers='host.docker.internal:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id='my-consumer-group',  #  IMPORTANT: no group = always read all messages
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
    value_deserializer=lambda v: v.decode('utf-8'),
    
    consumer_timeout_ms=5_000,  #  exit after 20 seconds if no messages
)


In [18]:
# get a single message from topic
#message = next(consumer)
#print(f"Key: {message.key}, Value: {message.value}")

Key: STOP, Value: STOP


### Attributes on KafkaConsumer:

    ConsumerRecord(
        topic, partition, leader_epoch, offset,
        timestamp, timestamp_type,
        key, value, headers, checksum,
        serialized_key_size, serialized_value_size, serialized_header_size
    )


In [33]:
# will listen forever without timeout - we workaround with using a STOP message for this example
for message in consumer:
    print(
        f"Key: {str(message.key):15s}, "
        f"Value: {message.value:20s}, "
        f"Timestamp: {message.timestamp}, "
        f"Offset: {message.offset}, "
    )
    consumer.commit()

    if message.key == "STOP":
        break
        pass

Key: None           , Value: Hello Kafka!        , Timestamp: 1779197167668, Offset: 24, 
Key: order123       , Value: Order Placed        , Timestamp: 1779197167671, Offset: 25, 
Key: None           , Value: Error Test          , Timestamp: 1779197167672, Offset: 26, 
Key: STOP           , Value: STOP                , Timestamp: 1779197167674, Offset: 27, 


In [65]:
# Send more messages
for i in range(10):
    producer.send(MY_TOPIC, key=f"order-x{i:02d}", value="Order Placed")
    time.sleep(1.1)
    print(".", end="")
producer.flush()
print("\nmessages sent!")

..........messages sent!


In [66]:
for i_poll in range(3):
    print(f"==== Poll #{i_poll+1} ===")
    result = consumer.poll(timeout_ms=5_000, max_records=4)
    for topic_partition, records in result.items():
        print( f"Topic-Partition -> topic: {topic_partition.topic}, partition: {topic_partition.partition}")
        for message in records:
            print(f"Record received -> Key: {message.key}, Value: {message.value}")
        print()

==== Poll #1 ===
Topic-Partition -> topic: my-topics, partition: 1
Record received -> Key: order-x02, Value: Order Placed
Record received -> Key: order-x04, Value: Order Placed
Record received -> Key: order-x05, Value: Order Placed
Record received -> Key: order-x06, Value: Order Placed

==== Poll #2 ===
Topic-Partition -> topic: my-topics, partition: 1
Record received -> Key: order-x07, Value: Order Placed

Topic-Partition -> topic: my-topics, partition: 0
Record received -> Key: order-x00, Value: Order Placed
Record received -> Key: order-x01, Value: Order Placed
Record received -> Key: order-x03, Value: Order Placed

==== Poll #3 ===
Topic-Partition -> topic: my-topics, partition: 0
Record received -> Key: order-x08, Value: Order Placed
Record received -> Key: order-x09, Value: Order Placed



In [78]:
help(list(result.values())[0][0])

Help on ConsumerRecord in module kafka.consumer.fetcher object:

class ConsumerRecord(builtins.tuple)
 |  ConsumerRecord(topic, partition, leader_epoch, offset, timestamp, timestamp_type, key, value, headers, checksum, serialized_key_size, serialized_value_size, serialized_header_size)
 |  
 |  ConsumerRecord(topic, partition, leader_epoch, offset, timestamp, timestamp_type, key, value, headers, checksum, serialized_key_size, serialized_value_size, serialized_header_size)
 |  
 |  Method resolution order:
 |      ConsumerRecord
 |      builtins.tuple
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __getnewargs__(self)
 |      Return self as a plain tuple.  Used by copy and pickle.
 |  
 |  __repr__(self)
 |      Return a nicely formatted representation string
 |  
 |  _asdict(self)
 |      Return a new dict which maps field names to their values.
 |  
 |  _replace(self, /, **kwds)
 |      Return a new ConsumerRecord object replacing specified fields with new values
 |  

In [1]:
from kafka import KafkaProducer
import time

producer = KafkaProducer(
    bootstrap_servers='host.docker.internal:9093',
    key_serializer=lambda k: k.encode('utf-8'),
    value_serializer=lambda v: v.encode('utf-8')
)

# Send messages
messages = [
    ('order1', 'Order Created'),
    ('order2', 'Order Paid'),
    ('order3', 'Order Shipped')
]

for key, value in messages:
    producer.send(MY_TOPIC, key=key, value=value)
    print(f"Sent -> Key: {key}, Value: {value}")
    time.sleep(0.5)

producer.flush()
producer.close()


Sent -> Key: order1, Value: Order Created
Sent -> Key: order2, Value: Order Paid
Sent -> Key: order3, Value: Order Shipped


In [49]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    MY_TOPIC,
    bootstrap_servers='host.docker.internal:9093',
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id="my-consumer-group2", #None,  #  IMPORTANT: no group = always read all messages
    consumer_timeout_ms=5_000,  #  exit after 5 seconds if no messages
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
    value_deserializer=lambda v: v.decode('utf-8')
)


In [57]:
message = next(consumer)
print(f"Received -> Key: {message.key}, Value: {message.value}")
consumer.commit()

StopIteration: 

In [6]:
for _ in range(14):
    try:
        message = next(consumer)
        print(f"Received -> Key: {message.key}, Value: {message.value}")
    except:
        pass
    
consumer.commit()

In [39]:
consumer.close()

In [44]:
consumer1 = KafkaConsumer(
    "my-topics" , #MY_TOPIC,
    bootstrap_servers='host.docker.internal:9093',
    #auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id="my-consumer-group2", #None,  #  IMPORTANT: no group = always read all messages
    consumer_timeout_ms=20_000,  #  exit after 5 seconds if no messages
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
    value_deserializer=lambda v: v.decode('utf-8')
)
consumer1

In [56]:
message = next(consumer1)
print(f"Received -> Key: {message.key}, Value: {message.value}")
consumer1.commit()

Received -> Key: None, Value: Error Test


In [7]:
for _ in range(3):
    try:
        message = next(consumer1)
        print(f"Received -> Key: {message.key}, Value: {message.value}")
    except:
        pass
    
consumer1.commit()

In [39]:
consumer1.close()

In [22]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    "my-topics" , #MY_TOPIC,
    bootstrap_servers='host.docker.internal:9093',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id=None,  #  IMPORTANT: no group = always read all messages
    consumer_timeout_ms=20_000,  #  exit after 5 seconds if no messages
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
    value_deserializer=lambda v: v.decode('utf-8')
)

print("Receiving messages...\n")

for message in consumer:
    print(f"Received -> Key: {message.key}, Value: {message.value}")

consumer.close()
print("\nDone reading messages.")


Receiving messages...

Received -> Key: order123, Value: Order Placed
Received -> Key: None, Value: Error Test
Received -> Key: None, Value: Hello Kafka!
Received -> Key: order123, Value: Order Placed
Received -> Key: None, Value: Error Test
Received -> Key: STOP, Value: STOP
Received -> Key: None, Value: Hello Kafka from additional producer!
Received -> Key: order, Value: Order Placed from additional producer
Received -> Key: None, Value: Error Test
Received -> Key: None, Value: Hello Kafka from additional producer!
Received -> Key: order, Value: Order Placed from additional producer
Received -> Key: None, Value: Error Test
Received -> Key: None, Value: Hello Kafka from additional producer!
Received -> Key: order, Value: Order Placed from additional producer
Received -> Key: None, Value: Error Test
Received -> Key: None, Value: Hello Kafka from additional producer!
Received -> Key: order, Value: Order Placed from additional producer
Received -> Key: None, Value: Error Test
Received ->